# Virtual Machines and KVM

Horacio Gomez-Acevedo

BMIG 50303

UAMS

>
**AI use**. Claude summarized the source webpages below for this session and the lecturer revised the content for accuracy.

**Sources and figures**
- Part I: Cloudzy Blog, *Virtual Machine (VM): Why and When Do You Need One?* (M. Schmitt, updated Jan 9, 2025) — https://cloudzy.com/blog/virtual-machine-vm-what-why-when/
- Part I: Oracle Virtual Box — https://www.virtualbox.org/manual/topics/Introduction.html#ct_about-virtualbox
- Part I: VMware, *What is a Virtual Machine?* — https://www.vmware.com/topics/virtual-machine
- Part II: Red Hat, *What is KVM?* (Apr 16, 2026) — https://www.redhat.com/en/topics/virtualization/what-is-KVM

**Part I — Introduction to Virtual Machines**
- I.1 What a virtual machine is
- I.2 Virtualization: history and types
- I.3 How VMs work: host, guest, hypervisor
- I.4 Advantages and limitations
- I.5 VMs vs. containers vs. VPS
- I.6 Common VM software

**Part II — Virtual Machines with KVM**
1. What KVM is
2. What a hypervisor does (host vs. guest)
3. How a VM runs under KVM
4. Benefits of virtualization
5. Key KVM features
6. Managing VMs (libvirt, virsh, virt-manager, Cockpit, KubeVirt)


# Part I — Introduction to Virtual Machines

## I.1 What is a virtual machine?

- A **virtual machine (VM)** is a complete, independent computer simulated in software. It can run its own operating system, for example in an app window on your desktop.
- The VM receives a share of the real machine's resources. **CPU, RAM, and storage limits can be set per VM.**
- Typical reasons to use one:
  - try another operating system without replacing your own
  - run applications your main OS can't run
  - test software in an isolated environment

## I.2 Virtualization: history and types

**Virtualization** means running one or more virtual instances on top of a physical computer.

- **1960s:** the idea started with mainframes, to split expensive hardware between several applications and users.
- **Before virtualization:** running Windows *and* Linux meant two machines (or dual-booting), with only one OS usable at a time.


<img src="../../Figures/fig_dual_boot.jpeg" width="400"/>

- **Local virtualization:** There is software that can run locally (e.g., Virtual box)

<img src="../../Figures/fig_virtual_box.png" width="400"/>

- **Today:** virtualization is the foundation of **cloud computing**, where it improves resource utilization.

Today, every component of a traditional data center or IT infrastructure can be virtualized. VMware groups this into **five types of virtualization**:

| Type | How it works | Main benefits |
|---|---|---|
| **1. Hardware (server)** | VMs are consolidated onto one physical server; a hypervisor talks directly to the server's CPU and disk to manage them | More efficient hardware use; one machine runs several OSes at once |
| **2. Software** | A full virtual computer lets a guest OS run on a host with a different OS (e.g., Android on Windows). Applications can also be virtualized and streamed from a server to laptops/phones | Run "foreign" OSes; remote access to centrally hosted apps |
| **3. Storage** | Many physical storage devices are pooled to look like a single device | Performance, load balancing, lower cost; easier disaster recovery (data duplicated and moved quickly) |
| **4. Network** | Equipment is combined into one software-based network, split into sub-networks; bandwidth is divided into independent channels assigned in real time | Reliability, speed, security, better usage monitoring; suits many always-connected users |
| **5. Desktop** | The desktop environment is separated from the device and stored on a remote server | Access from anywhere on any device; data security; lower license/update costs; easier management |



## I.3 How VMs work: host, guest, hypervisor

- The **host** is the physical computer; each VM's OS is a **guest**.
- A VM runs as a **process** on the host, like any other application.
- Changes inside a guest **don't affect the host or other guests** (isolation).
- A VM is stored as a set of files, typically: **configuration file, virtual disk file, NVRAM (firmware settings) file, log file**.

<img src="../../Figures/fig_hypervisor.jpeg" width="400"/>


### The hypervisor
A VM cannot access physical hardware directly. The **hypervisor** (or *virtual machine monitor*) is the software layer between them. It:
- presents virtual hardware (CPU, memory, disk, I/O) to each guest
- allocates the host's real resources among guests
- keeps VMs isolated from one another

| | **Type 1 — bare metal** | **Type 2 — hosted** |
|---|---|---|
| Runs on | Directly on the hardware | On top of a host OS |
| Typical use | Data centers, cloud, servers | Desktops, labs, testing |
| Examples | Xen, VMware ESXi, Citrix Hypervisor, **KVM** | VirtualBox, VMware Workstation |



```
   Type 1 (bare metal)              Type 2 (hosted)
+-------+-------+-------+      +-------+-------+
| Guest | Guest | Guest |      | Guest | Guest |  apps
+-------+-------+-------+      +-------+-------+------+
|      Hypervisor       |      |  Hypervisor   |      |
+-----------------------+      +---------------+------+
|       Hardware        |      |      Host OS         |
+-----------------------+      +----------------------+
                               |      Hardware        |
                               +----------------------+
```

## I.4 Advantages and limitations

| Advantages | Limitations |
|---|---|
| **Isolation:** workloads don't conflict with each other or the host | **Overhead:** hardware access goes through the hypervisor, so it is slower than bare metal |
| **Try other OSes safely** (including betas; malware analysis) | **Resource consumption:** many VMs can exhaust host CPU/RAM; poor allocation can affect the host |
| **Developer friendly:** reusable templates and snapshots | **Graphics-heavy apps & games** perform poorly |
| **Flexibility:** create and resize VMs quickly | **Storage:** each VM's virtual disk takes significant space |
| **Migration:** move VMs between hosts, or on-premises ↔ cloud | **Licensing:** guest OSes and software still need licenses |
| **Cost-effective:** consolidate many servers onto fewer machines | |

## I.5 VMs vs. containers vs. VPS

**Containers** do not use a hypervisor or virtual hardware. They package an application with its libraries and binaries, and all containers on a host **share the host's kernel**. Containers often run *inside* VMs.

| | Virtual machine | Container |
|---|---|---|
| Isolation unit | Whole computer (own kernel + OS) | Application process |
| Kernel | Own guest kernel | Shared host kernel |
| Size / boot time | GBs / minutes | MBs / seconds |
| Runs a different OS than host? | Yes | No (same kernel family) |
| Typical use | Full OS environments, legacy apps, strong isolation | Microservices, web servers, CI/DevOps |

**VPS (Virtual Private Server):** a VM rented from a hosting provider. A physical server is divided into virtual servers, each with its own root access and full control. Technically, a VPS and a cloud VM use the same virtualization technology.

## I.6 Common VM software

| Software | Hypervisor type | Notes |
|---|---|---|
| Oracle VirtualBox | Type 2 | Free; popular for desktops and teaching |
| VMware Workstation | Type 2 | Desktop virtualization |
| **KVM (+ QEMU)** | Type 1 (in Linux kernel) | Linux's built-in hypervisor (Part II) |
| Xen | Type 1 | Open source; used by early public clouds |
| Citrix Hypervisor (XenServer) | Type 1 | Enterprise product based on Xen |

**Part I takeaways**
1. A VM is a software computer; the **hypervisor** shares the real hardware among isolated guests.
2. **Type 1** hypervisors run on the hardware; **Type 2** run on a host OS.
3. VMs give isolation and flexibility, at the cost of overhead and resources.
4. **Containers** are lighter but share the host kernel; VMs run complete operating systems.

---
# Part II — Virtual Machines with KVM

## 1. What is KVM?

- **Kernel-based Virtual Machine (KVM)** is an open source virtualization technology built into Linux.
- KVM turns **Linux itself into a hypervisor** that runs multiple isolated virtual machines (VMs).
- Timeline: announced in **2006**, merged into the mainline Linux kernel **one year later (2007)**.
- Many open source virtualization stacks — including Red Hat's virtualization portfolio — use KVM as their hypervisor.

## 2. What does a hypervisor do?

A hypervisor **pools** compute resources (CPU, memory, storage) and **reallocates** them among VMs. It can run, manage, and create VMs.

| Term | Meaning |
|---|---|
| **Host** | The physical machine providing resources |
| **Guest** | A VM consuming the host's resources |

To run VMs, a hypervisor needs operating-system-level components. **KVM already has them because it is part of the Linux kernel:**

In [2]:
components = {
    "Memory manager":    "allocates RAM to guests",
    "Process scheduler": "decides which VM (process) gets CPU time",
    "I/O stack":         "handles disk and device input/output",
    "Device drivers":    "talk to physical hardware",
    "Security manager":  "isolates guests (SELinux / sVirt)",
    "Network stack":     "provides guest networking",
}
print(f"{'OS component':<20} | Role in virtualization")
print("-" * 60)
for name, role in components.items():
    print(f"{name:<20} | {role}")

OS component         | Role in virtualization
------------------------------------------------------------
Memory manager       | allocates RAM to guests
Process scheduler    | decides which VM (process) gets CPU time
I/O stack            | handles disk and device input/output
Device drivers       | talk to physical hardware
Security manager     | isolates guests (SELinux / sVirt)
Network stack        | provides guest networking


## 3. How a VM runs under KVM

> *"A VM under KVM is a Linux process, scheduled and managed by the kernel."*

- Each VM is implemented as a **regular Linux process**.
- It is scheduled by the **standard Linux scheduler**, so it inherits Linux performance features and fine-grained resource control.
- Each VM gets dedicated **virtual hardware**: CPUs, memory, disks, network card, graphics adapter.

```
+--------------------------------------------------------------+
|                     Physical host hardware                   |
+--------------------------------------------------------------+
|        Linux kernel  +  KVM module  (= the hypervisor)       |
|  scheduler | memory mgr | I/O stack | drivers | SELinux/sVirt|
+---------------+---------------+---------------+--------------+
|  VM 1 process |  VM 2 process |  VM 3 process | normal apps  |
|  vCPU, vRAM,  |  vCPU, vRAM,  |  vCPU, vRAM,  |              |
|  vDisk, vNIC  |  vDisk, vNIC  |  vDisk, vNIC  |              |
|  Guest OS     |  Guest OS     |  Guest OS     |              |
+---------------+---------------+---------------+--------------+
```

**Try it on a Linux machine** — check whether KVM is available:

In [3]:
import os
if os.path.exists("/dev/kvm"):
    print("/dev/kvm found: KVM hardware virtualization is available.")
else:
    print("/dev/kvm not found: KVM is not available here (no CPU support, module not loaded, or running in a container).")

/dev/kvm not found: KVM is not available here (no CPU support, module not loaded, or running in a container).


## 4. Benefits of virtualization

| Benefit | What it means |
|---|---|
| **Flexibility** | Divide one host into many VMs; allocate resources where needed — saves space, power, and maintenance |
| **Speed** | VMs are defined in software: quick to create, clone, remove, migrate; can be automated and controlled remotely |
| **Compatibility** | Run software not built for the host OS (e.g., keep a legacy OS alive on modern hardware) |
| **Stability & safety** | VMs are isolated — one unstable VM doesn't affect the host or others; safe for testing new configs |

## 5. Key KVM features

### 5.1 Security — SELinux + sVirt
- **SELinux** draws security boundaries around each VM.
- **sVirt** extends SELinux to apply **Mandatory Access Control (MAC)** to guests and prevents manual labeling errors.

### 5.2 Storage flexibility
- Any storage Linux supports: local disks, network-attached storage (NAS).
- **Shared file systems** let multiple hosts share VM images.

### 5.3 Hardware support
In **RHEL 9**, KVM is supported on 64-bit **AMD, Intel, ARM**, and **IBM z13 and later**.

### 5.4 Live migration
- Move a **running** VM between physical hosts with no noticeable interruption: VM stays on, network connections stay up, apps keep running.
- KVM can also **save a VM's state** and resume it later.

## 6. Managing VMs with KVM

| Tool | Type | Use |
|---|---|---|
| **libvirt** | API / library | Common management API for virtualization platforms |
| **virsh** | CLI (libvirt) | Create, start, list, stop VMs; virtualization shell |
| **virt-manager** (VMM) | Desktop GUI | Point-and-click VM management on Linux desktops |
| **Cockpit** | Web console | Browser-based VM management (RHEL virtualization plug-in) |
| **KubeVirt** | Kubernetes add-on | Run VMs alongside containers at scale; basis of OpenShift Virtualization |

Common `virsh` commands (run on a KVM host, shown here for reference):

```bash
virsh list --all          # list all VMs (running and stopped)
virsh start  myvm         # boot a VM
virsh shutdown myvm       # graceful shutdown
virsh destroy  myvm       # force power-off
virsh dominfo  myvm       # vCPUs, memory, state
virsh migrate --live myvm qemu+ssh://host2/system   # live migration
virsh save myvm myvm.state ; virsh restore myvm.state  # save / resume state
```

---
### Key takeaways
1. KVM makes the **Linux kernel the hypervisor** — no separate hypervisor OS needed.
2. **VMs are Linux processes**, so they reuse Linux's scheduler, memory manager, drivers, and security.
3. Strong features: **SELinux/sVirt security, flexible storage, broad hardware support, live migration**.
4. Managed via **libvirt/virsh, virt-manager, Cockpit**, or **KubeVirt** at Kubernetes scale.

*Further reading (Red Hat):* KVM vs. VMware — https://www.redhat.com/en/topics/virtualization/kvm-vs-vmware-comparison